In [1]:
import zipfile
from pathlib import Path
import pandas as pd

raw_dir = Path("/home/eprashar_solutions_corelogic_com/network-idx/data/raw/fcc/broadband_coverage")
extract_dir = Path("/home/eprashar_solutions_corelogic_com/network-idx/data/extracted/fcc/broadband_coverage")
extract_dir.mkdir(parents=True, exist_ok=True)


In [7]:
# Grab the first zip file in the directory
zip_file = next(raw_dir.glob("*.zip"))
print(f"Extracting: {zip_file.name}")

with zipfile.ZipFile(zip_file, "r") as zf:
    print("Contents:", zf.namelist())
    csv_name = [n for n in zf.namelist() if n.endswith(".csv")][0]
    zf.extract(csv_name, path=extract_dir)

csv_path = extract_dir / csv_name
df = pd.read_csv(csv_path, nrows=100)
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

Extracting: bdc_01_fixed_broadband_summary_by_geography_place_J25_19mar2026.zip
Contents: ['bdc_01_fixed_broadband_summary_by_geography_place_J25_19mar2026.csv']
Shape: (100, 14)
Columns: ['area_data_type', 'geography_type', 'geography_id', 'geography_desc', 'geography_desc_full', 'total_units', 'biz_res', 'technology', 'speed_02_02', 'speed_10_1', 'speed_25_3', 'speed_100_20', 'speed_250_25', 'speed_1000_100']


,area_data_type,geography_type,geography_id,geography_desc,geography_desc_full,total_units,biz_res,technology,speed_02_02,speed_10_1,speed_25_3,speed_100_20,speed_250_25,speed_1000_100
0,Total,Census Place,100100,Abanda,"Abanda, AL",97,R,Any Technology,1.000000,1.0,1.0,1.0,1.0,0.0
1,Total,Census Place,100100,Abanda,"Abanda, AL",97,B,Any Technology,1.000000,1.0,1.0,1.0,1.0,0.0
2,Total,Census Place,100100,Abanda,"Abanda, AL",97,R,All Wired,0.257732,0.0,0.0,0.0,0.0,0.0
3,Total,Census Place,100100,Abanda,"Abanda, AL",97,B,All Wired,0.257732,0.0,0.0,0.0,0.0,0.0
4,Total,Census Place,100100,Abanda,"Abanda, AL",97,R,Any Terrestrial,0.257732,0.0,0.0,0.0,0.0,0.0


In [3]:
# Load parquet file
parquet_path = Path(r"/home/eprashar_solutions_corelogic_com/network-idx/data/processed/fcc/broadband_coverage/fcc_fixed_coverage_AK_02.parquet")
df = pd.read_parquet(parquet_path)
df.head()

,geography_id,geography_desc,geography_desc_full,total_units,copper_speed_100_20,copper_less_than_100_20,copper_more_than_100_20,cable_speed_100_20,cable_less_than_100_20,cable_more_than_100_20,fiber_speed_100_20,fiber_less_than_100_20,fiber_more_than_100_20
0,200065,Adak,"Adak, AK",407,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.498771,0.000000
1,200650,Akhiok,"Akhiok, AK",47,0.0,0.617021,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000
2,200760,Akiachak,"Akiachak, AK",181,0.0,0.983425,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000
3,200870,Akiak,"Akiak, AK",130,0.0,0.953846,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000
4,201090,Akutan,"Akutan, AK",104,0.0,0.000000,0.0,0.0,0.0,0.0,0.980769,0.980769,0.980769


In [2]:
# Load provider parquet
parquet_path = Path("/home/eprashar_solutions_corelogic_com/network-idx/data/processed/fcc/speeds/fcc_fixed_speeds_providers_AK_02.parquet")
df = pd.read_parquet(parquet_path)
df.head()

,state_usps,state_fips,block_geoid,frn,provider_id,brand_name,cable_location_count,cable_max_download_speed,cable_max_upload_speed,copper_location_count,copper_max_download_speed,copper_max_upload_speed,fiber_location_count,fiber_max_download_speed,fiber_max_upload_speed
0,AK,02,020130001001030,0003739521,131226,Fastwyre Broadband,NaN,NaN,NaN,1.0,0.0,0.0,NaN,NaN,NaN
1,AK,02,020130001001053,0003739521,131226,Fastwyre Broadband,NaN,NaN,NaN,9.0,0.0,0.0,NaN,NaN,NaN
2,AK,02,020130001001076,0003739521,131226,Fastwyre Broadband,NaN,NaN,NaN,2.0,0.0,0.0,NaN,NaN,NaN
3,AK,02,020130001001166,0003739521,131226,Fastwyre Broadband,NaN,NaN,NaN,17.0,0.0,0.0,NaN,NaN,NaN
4,AK,02,020130001001168,0003739521,131226,Fastwyre Broadband,NaN,NaN,NaN,9.0,0.0,0.0,NaN,NaN,NaN


In [11]:
# Cell 2 — Aggregate by provider across the country
TECH_METRICS = {
    "cable": ["cable_location_count", "cable_max_download_speed", "cable_max_upload_speed"],
    "copper": ["copper_location_count", "copper_max_download_speed", "copper_max_upload_speed"],
    "fiber": ["fiber_location_count", "fiber_max_download_speed", "fiber_max_upload_speed"],
}

GROUP_COLS = [
    #"frn", 
    "provider_id",
    "brand_name"]

agg_dict = {}
for tech, cols in TECH_METRICS.items():
    loc_col, dl_col, ul_col = cols
    agg_dict[loc_col] = (loc_col, "sum")
    agg_dict[dl_col] = (dl_col, "max")
    agg_dict[ul_col] = (ul_col, "max")

# Also count how many states each provider operates in
agg_dict["state_count"] = ("state_usps", "nunique")

providers = df.groupby(GROUP_COLS, as_index=False).agg(**agg_dict)
print(f"Unique providers: {len(providers):,}")
providers.sort_values("cable_location_count", ascending=False, na_position="last").head(20)

Unique providers: 2,153


,provider_id,brand_name,cable_location_count,cable_max_download_speed,cable_max_upload_speed,copper_location_count,copper_max_download_speed,copper_max_upload_speed,fiber_location_count,fiber_max_download_speed,fiber_max_upload_speed,state_count
224,130317,Xfinity,37771116.0,2000.0,2000.0,0,NaN,NaN,872882.0,10000.0,10000.0,49
166,130235,Spectrum,34927455.0,1000.0,1000.0,0,NaN,NaN,2175309.0,1000.0,1000.0,42
242,130360,Cox Communications,6746045.0,2000.0,500.0,3.0,500.0,10.0,784149.0,2000.0,2000.0,20
251,130370,Optimum,5633338.0,940.0,35.0,0,NaN,NaN,2323119.0,8000.0,8000.0,21
552,130804,Mediacom Xtream,2600711.0,1000.0,50.0,0,NaN,NaN,13360.0,2000.0,1000.0,22
48,130079,Astound Broadband,2304242.0,1500.0,50.0,13122.0,1000.0,10.0,292155.0,5000.0,5000.0,13
129,130183,Sparklight,1773528.0,1000.0,50.0,0,NaN,NaN,105148.0,1000.0,1000.0,22
1072,131480,"WOW Internet, Cable & Phone",1647538.0,5000.0,5000.0,5509.0,25.0,3.0,116151.0,5000.0,5000.0,6
1398,280001,Breezeline,1248089.0,1000.0,1000.0,0,NaN,NaN,71317.0,1000.0,1000.0,13
490,130741,Liberty,972186.0,1000.0,30.0,0,NaN,NaN,201661.0,1000.0,1000.0,1


In [12]:
# Cell 4 — Technology mix: how many providers offer each tech?
print("Providers with cable coverage:", providers["cable_location_count"].notna().sum())
print("Providers with copper coverage:", providers["copper_location_count"].notna().sum())
print("Providers with fiber coverage:", providers["fiber_location_count"].notna().sum())

Providers with cable coverage: 2153
Providers with copper coverage: 2153
Providers with fiber coverage: 2153
